In [1]:
# Importando bibliotecas

import numpy as np
import scipy as sc
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy.stats import skew, kurtosis

In [2]:
path_healthy = 'D:\Artigo\Healthy\Healthy' #Caminho para acessar os dados saudáveis

In [3]:
path_damaged = 'D:\Artigo\Damaged\Damaged' #Caminho para acessar os dados danificados

# Todos os minutos

In [4]:
# Função para preparar dados saudáveis
def prepare_healthy(path):
    files = ['H1.mat', 'H2.mat', 'H3.mat', 'H4.mat', 'H5.mat', 'H6.mat', 'H7.mat', 'H8.mat', 'H9.mat', 'H10.mat'] #Lista com nomes dos arquivos saudáveis
    list_df_healthy = [] #Cria lista vazia
    for i in files:
        file_path = os.path.join(path, i) #Criando o caminho com a pasta e o nome do arquivo
        H = sc.io.loadmat(file_path) #Abre o arquivo .mat
        H = {k: v for k, v in H.items() if not k.startswith('__')} #Seleciona as colunas com dados
        df = pd.DataFrame({k: v.squeeze() for k, v in H.items()}) #Transforma em dataframe pandas
        df = df.drop(columns=['Speed']) #Remove coluna Speed
        list_df_healthy.append(df) #Adiciona o dataframe na lista
        print(i)
    return list_df_healthy #Retorna lista com os dataframes por minuto

In [6]:
# Função para preparar dados danificados
def prepare_damaged(path):
    files = ['D1.mat', 'D2.mat', 'D3.mat', 'D4.mat', 'D5.mat', 'D6.mat', 'D7.mat', 'D8.mat', 'D9.mat', 'D10.mat'] #Lista com nomes dos arquivos danificados
    list_df_damaged = [] #Cria lista vazia
    for i in files:
        file_path = os.path.join(path, i) #Criando o caminho com a pasta e o nome do arquivo
        D = sc.io.loadmat(file_path) #Abre o arquivo .mat
        D = {k: v for k, v in D.items() if not k.startswith('__')} #Seleciona as colunas com dados
        df = pd.DataFrame({k: v.squeeze() for k, v in D.items()}) #Transforma em dataframe pandas
        df = df.drop(columns=['Speed', 'Torque']) #Remove as colunas Speed e Torque
        list_df_damaged.append(df) #Adiciona o dataframe na lista
        print(i)
    return list_df_damaged #Retorna lista com os dataframes por minuto

In [7]:
healthy = prepare_healthy(path_healthy) #Chama função e cria lista com dataframes dos dados saudáveis

H1.mat
H2.mat
H3.mat
H4.mat
H5.mat
H6.mat
H7.mat
H8.mat
H9.mat
H10.mat


In [8]:
damaged = prepare_damaged(path_damaged) #Chama função e cria lista com dataframes dos dados danificados

D1.mat
D2.mat
D3.mat
D4.mat
D5.mat
D6.mat
D7.mat
D8.mat
D9.mat
D10.mat


In [9]:
df_healthy = pd.concat(healthy) #Junta os dataframes saudáveis em um só

In [10]:
df_damaged = pd.concat(damaged) #Junta os dataframes danificados em um só

In [11]:
df_join = pd.concat([df_healthy, df_damaged]) #Junta os dataframes danificado e saudável

In [12]:
def statistic_summary(df):

    # Filtrando apenas colunas numéricas
    numeric_df = df.select_dtypes(include=[np.number])

    # Calculando estatísticas básicas
    statistics_summary = numeric_df.describe().T

    # Calculando estatísticas adicionais
    statistics_summary['median'] = numeric_df.median()
    statistics_summary['mode'] = numeric_df.mode().iloc[0]
    statistics_summary['range'] = numeric_df.max() - numeric_df.min()
    statistics_summary['cv'] = statistics_summary['std'] / statistics_summary['mean']  # Coeficiente de variação
    statistics_summary['skewness'] = numeric_df.apply(lambda x: skew(x.dropna()))
    statistics_summary['kurtosis'] = numeric_df.apply(lambda x: kurtosis(x.dropna()))
    statistics_summary['sem'] = numeric_df.sem()  # Erro padrão da média

    # Calculando quartis e intervalo interquartil
    quartiles = numeric_df.quantile([0.25, 0.75])
    statistics_summary['Q1'] = quartiles.loc[0.25]
    statistics_summary['Q3'] = quartiles.loc[0.75]
    statistics_summary['IQR'] = statistics_summary['Q3'] - statistics_summary['Q1']

    # Visualizando o resumo das estatísticas
    statistics_summary = statistics_summary[['count', 'mean', 'std', 'median', 'mode', 'min', 'max', 'range', 'Q1', 'Q3', 'IQR', 'cv', 'skewness', 'kurtosis', 'sem']]

    # Formatando todas as colunas para exibir duas casas decimais
    statistics_summary = statistics_summary.applymap(lambda x: f"{x:.2f}")

    return statistics_summary

In [15]:
from scipy import stats

def statistic_summary_numpy(df):

    # Filtrando apenas colunas numéricas
    df_array = df.to_numpy().T
    cols = df.columns

    basic_stats = {}

    # Calculando estatísticas básicas
    for i in range(len(df_array)):
        count = len(df_array[i]) #Calcula contagem de dados
        mean = np.mean(df_array[i]) #Calcula média
        std = np.std(df_array[i]) #Calcula desvio padrão
        median = np.median(df_array[i]) #Calcula mediana
        mode = stats.mode(df_array[i])[0][0] #Calcula moda
        min_l = np.min(df_array[i]) #Calcula valor mínimo
        max_l = np.max(df_array[i]) # Calcula valor máximo
        range_l = np.max(df_array[i]) - np.min(df_array[i]) #Calcula alcance dos dados
        quantiles = np.quantile(df_array[i], [0.25, 0.75]) #Calcula os quartis
        q1 = quantiles[0] #Primeiro quartil
        q3 = quantiles[1] #Terceiro quartil
        iqr = q3-q1 #Calcula o IQR
        cv = std/mean #Calcula o coeficiente de variação
        skew_l = skew(df_array[i]) #Calcula o skewness
        kurt_l = kurtosis(df_array[i]) #Calcula kurtosis
        sem = stats.sem(df_array[i]) #Calcula erro padrão da média

        stats_list = [count, mean, std, median, mode, min_l, max_l, range_l, q1, q3, iqr, cv, skew_l, kurt_l, sem] #Lista com as estatísticas
        round_list = [round(x, 2) for x in stats_list] #Arredonda os valores para duas casas decimais

        basic_stats[cols[i]] = round_list

    statistics_summary_header = ['count', 'mean', 'std', 'median', 'mode', 'min', 'max', 'range', 'Q1', 'Q3', 'IQR', 'cv', 'skewness', 'kurtosis', 'sem'] #Nomes das Colunas

    statistics_summary = pd.DataFrame(basic_stats) #Cria o datafeame

    statistics_summary = statistics_summary.T
    statistics_summary.columns = statistics_summary_header #Define os nomes das colunas

    return statistics_summary

In [16]:
summary_np = statistic_summary_numpy(df_join) #Chama função e cria daframe com estatísticas

C:\Users\Jorge\AppData\Local\Temp\ipykernel_13468\2763141905.py:17: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode = stats.mode(df_array[i])[0][0] #Calcula moda


In [17]:
summary_np

,count,mean,std,median,mode,min,max,range,Q1,Q3,IQR,cv,skewness,kurtosis,sem
AN3,48000000.0,-0.02,2.17,-0.03,0.12,-12.12,12.55,24.67,-1.18,1.14,2.32,-126.87,0.05,1.26,0.0
AN4,48000000.0,-0.01,2.41,0.02,0.26,-14.50,18.50,33.00,-1.58,1.57,3.15,-220.23,-0.05,0.37,0.0
AN5,48000000.0,-0.11,3.05,-0.14,0.27,-19.94,22.37,42.31,-1.80,1.52,3.33,-27.14,0.12,1.61,0.0
AN6,48000000.0,-0.14,3.64,-0.17,0.00,-21.72,23.02,44.74,-2.48,2.17,4.65,-25.88,0.03,0.44,0.0
AN7,48000000.0,-0.02,3.35,-0.03,-0.31,-20.79,21.41,42.21,-2.17,2.11,4.27,-167.28,0.03,0.45,0.0
AN8,48000000.0,-0.06,6.51,-0.00,0.31,-45.28,41.16,86.44,-4.19,4.12,8.31,-101.21,-0.06,0.61,0.0
AN9,48000000.0,-0.23,5.31,-0.32,-0.51,-35.53,35.86,71.39,-2.69,2.06,4.74,-22.95,0.14,2.01,0.0
AN10,48000000.0,-0.01,3.16,-0.01,-0.14,-22.51,19.11,41.62,-1.80,1.80,3.60,-521.76,-0.05,1.15,0.0


In [ ]:
summary_np.to_csv('Statistics_10min.csv',index=False) #Exporta as estatísticas de todos os mintuos em formato .csv

# Primeiro minuto saudável + danificado

In [ ]:
# Função para preparar dados saudáveis
def prepare_healthy(path):
    files = ['H1.mat'] #Lista com arquivo do primeiro minuto dos dados saudáveis
    list_df_healthy = [] #Cria lista vazia
    for i in files:
        file_path = os.path.join(path, i) #Criando o caminho com a pasta e o nome do arquivo
        H = sc.io.loadmat(file_path) #Abre o arquivo .mat
        H = {k: v for k, v in H.items() if not k.startswith('__')} #Seleciona as colunas com dados
        df = pd.DataFrame({k: v.squeeze() for k, v in H.items()}) #Transforma em dataframe pandas
        df = df.drop(columns=['Speed']) #Remove coluna Speed
        list_df_healthy.append(df) #Adiciona o dataframe na lista
        print(i)
    return list_df_healthy #Retorna lista com o dataframe

In [ ]:
# Função para preparar dados danificados
def prepare_damaged(path):
    files = ['D1.mat'] #Lista com arquivo do primeiro minuto dos dados danificados
    list_df_damaged = [] #Cria lista vazia
    for i in files:
        file_path = os.path.join(path, i) #Criando o caminho com a pasta e o nome do arquivo
        D = sc.io.loadmat(file_path) #Abre o arquivo .mat
        D = {k: v for k, v in D.items() if not k.startswith('__')} #Seleciona as colunas com dados
        df = pd.DataFrame({k: v.squeeze() for k, v in D.items()}) #Transforma em dataframe pandas
        df = df.drop(columns=['Speed', 'Torque']) #Remove as colunas Speed e Torque
        list_df_damaged.append(df) #Adiciona o dataframe na lista
        print(i)
    return list_df_damaged #Retorna lista com o dataframe

In [20]:
healthy_1 = prepare_healthy(path_healthy) #Chama função e cria lista com dataframes dos dados saudáveis

H1.mat


In [21]:
damaged_1 = prepare_damaged(path_damaged) #Chama função e cria lista com dataframes dos dados danificados

D1.mat


In [22]:
df_join_1 = pd.concat([healthy_1[0], damaged_1[0]]) #Junta os dataframes danificado do primeiro mintuo e saudável do primeiro minuto

In [24]:
summary_1 = statistic_summary_numpy(df_join_1) #Cria dataframe com estatísticas para os primeiros minutos

C:\Users\Jorge\AppData\Local\Temp\ipykernel_13468\2763141905.py:17: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode = stats.mode(df_array[i])[0][0] #Calcula moda


In [25]:
summary_1

,count,mean,std,median,mode,min,max,range,Q1,Q3,IQR,cv,skewness,kurtosis,sem
AN3,4800000.0,-0.02,2.19,-0.03,-0.09,-11.35,11.82,23.17,-1.19,1.13,2.32,-131.91,0.08,1.28,0.0
AN4,4800000.0,-0.01,2.47,0.03,0.02,-12.87,13.22,26.10,-1.59,1.59,3.18,-234.02,-0.06,0.46,0.0
AN5,4800000.0,-0.11,2.98,-0.14,-0.31,-19.94,22.37,42.31,-1.77,1.50,3.28,-27.13,0.11,1.73,0.0
AN6,4800000.0,-0.14,3.47,-0.17,0.00,-18.22,18.84,37.05,-2.41,2.10,4.51,-25.04,0.04,0.30,0.0
AN7,4800000.0,-0.02,3.25,-0.04,-0.21,-17.37,17.89,35.26,-2.13,2.07,4.19,-176.59,0.05,0.36,0.0
AN8,4800000.0,-0.06,6.31,-0.01,0.38,-33.29,35.60,68.89,-4.12,4.05,8.18,-100.83,-0.05,0.49,0.0
AN9,4800000.0,-0.23,5.28,-0.30,-0.87,-33.07,34.00,67.07,-2.75,2.13,4.88,-22.95,0.13,1.85,0.0
AN10,4800000.0,-0.01,3.11,-0.02,0.30,-19.33,17.07,36.40,-1.79,1.78,3.57,-524.34,-0.02,1.06,0.0


In [ ]:
summary_1.to_csv('Statistics_1min.csv',index=False) #Exporta as estatísticas do primeiro minuto em formato .csv